# M2 Kelompok 03 — peluncur Colab CPU
Jalankan **per sel**, bukan Run all. Upload `src/M2_Encoders.py` lokal melalui sel 3 karena berkas ini **belum di-push**. Dataset ZIP di Drive mengikuti path notebook M1. Hasil M1 di repo hanya dibaca. Tidak ada eksekusi otomatis saat notebook dibuka.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
print('Python:', sys.version)
available_kb = int(next(line.split()[1] for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemAvailable:')))
print('MemAvailable (GB):', round(available_kb / 1e6, 2))
print('Free disk (GB):', round(shutil.disk_usage('/content').free / 1e9, 2))
assert available_kb >= 6_000_000, 'RAM tersedia kurang dari 6 GB; jangan jalankan encoder'
assert shutil.disk_usage('/content').free >= 10_000_000_000, 'Ruang Colab kurang dari 10 GB'


In [ ]:
repo = Path('/content/Tubes-Multimodal')
assert not repo.exists(), 'Gunakan runtime Colab baru agar repo bersih; jangan menimpa folder yang sudah ada.'
subprocess.run(['git', 'clone', 'https://github.com/bayybrigas04/Tubes-Multimodal.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo / 'requirements.txt'), 'open_clip_torch', 'torchvision'], check=True)
print('Commit repo:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
from google.colab import files
print('Pilih Q:\\Tubes_Multimodal\\github_k3\\src\\M2_Encoders.py dari komputer Mak')
uploaded = files.upload()
assert list(uploaded) == ['M2_Encoders.py'], 'Unggah hanya M2_Encoders.py dengan nama persis; ulangi sel pada runtime baru bila keliru'
(repo / 'src' / 'M2_Encoders.py').write_bytes(uploaded['M2_Encoders.py'])
import ast
ast.parse((repo / 'src' / 'M2_Encoders.py').read_text(encoding='utf-8'))
print('Kode M2 terunggah; model belum dijalankan.')


In [ ]:
zip_path = Path('/content/drive/MyDrive/Dataset-Tubes-Multimodal/bundle-mahasiswa.zip')
assert zip_path.is_file(), f'ZIP tidak ada: {zip_path}'
data_root = Path('/content/data')
subprocess.run(['unzip', '-q', str(zip_path), '-d', str(data_root)], check=True)
data_dir = data_root / 'bundle-mahasiswa'
assert all((data_dir / filename).is_file() for filename in ('train.csv', 'val.csv')), f'Struktur ZIP tidak sesuai: {data_dir}'
assert (data_dir / 'images').is_dir()
print('Dataset:', data_dir)


## Gate sebelum E1
Pastikan RAM tersedia dan disk memadai. Perintah berikut **mengunduh model sekitar 1,5 GB dan menjalankan E1 pada val**; tidak melatih proyeksi. Lihat hasil dan simpan sebelum memutuskan E2.


In [ ]:
import datetime
run_name = 'e1-val-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_dir = Path('/content/drive/MyDrive/Dataset-Tubes-Multimodal/m2-runs') / run_name
assert not out_dir.exists()
command = [sys.executable, str(repo / 'src' / 'M2_Encoders.py'), '--data', str(data_dir), '--out', str(out_dir), '--baseline', str(repo / 'results/m1/tabel_recall_val.csv'), '--e1-revision', '75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2', '--e2-revision', 'a7c4a9bf24872c73a2c059ab2f9e9123a4d4144e', '--models', 'E1', '--skip-projection', '--epochs', '5', '--lr', '0.001', '--batch-size', '4', '--seed', '42']
print('Output:', out_dir)
subprocess.run(command, check=True)
print((out_dir / 'm2_val.csv').read_text(encoding='utf-8'))


**Berhenti di sini dahulu.** Kirim `m2_val.csv`, `costs.json`, `manifest.json` atau traceback kepada Cauw sebelum melanjutkan E2/proyeksi. Jangan menganggap angka estimasi sebelumnya sebagai waktu terukur.
